# Debug Flow - RAG System
Este notebook testa cada componente do sistema RAG para identificar erros

In [20]:
import os
from dotenv import load_dotenv
import sys


sys.path.append(os.path.abspath('.'))

load_dotenv()
print("✅ Ambiente carregado")

✅ Ambiente carregado


## 1. Teste do PdfProcessor

In [21]:
from rag.core.PdfProcessor import PdfProcessor
import os

file_path = os.getenv("FILE_PATH")
print(f"📄 PDF Path: {file_path}")
print(f"📁 Arquivo existe? {os.path.exists(file_path)}")

if not os.path.exists(file_path):
    print("❌ ERRO: Arquivo não encontrado!")
else:
    processor = PdfProcessor()
    print(f"✅ PdfProcessor criado")
    print(f"   - chunk_size: {processor.text_splitter._chunk_size}")
    print(f"   - chunk_overlap: {processor.text_splitter._chunk_overlap}")

📄 PDF Path: rag/data/documents/PPCENGSOFTWARE.pdf
📁 Arquivo existe? True
✅ PdfProcessor criado
   - chunk_size: 100
   - chunk_overlap: 20


## 2. Processar PDF

In [22]:
try:
    chunks = processor.process_pdf(file_path)
    print(f"✅ PDF processado com sucesso!")
    print(f"   - Total de chunks: {len(chunks)}")
    print(f"\n📝 Exemplo de chunk:")
    print(f"   Conteúdo: {chunks[0].page_content[:200]}...")
    print(f"   Metadata: {chunks[0].metadata}")
except Exception as e:
    print(f"❌ ERRO ao processar PDF: {e}")
    import traceback
    traceback.print_exc()

✅ PDF processado com sucesso!
   - Total de chunks: 3974

📝 Exemplo de chunk:
   Conteúdo: Projeto Pedagógico do Curso Bacharelado em Engenharia de Software Modalidade: presencial 4 IFSP SÃO...
   Metadata: {'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'rag/data/documents/PPCENGSOFTWARE.pdf', 'file_path': 'rag/data/documents/PPCENGSOFTWARE.pdf', 'total_pages': 151, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 1, 'chunk_method': 'smart_pdf_processor', 'char_count': 328}


## 3. Teste do VectorStoreIngestor

In [23]:
from rag.core.VectorStoreIngestor import VectorStoreIngestor

persist_dir = "rag/vector_store/db/chroma"
print(f"📁 Persist dir: {persist_dir}")
print(f"📁 Diretório existe? {os.path.exists(persist_dir)}")

try:
    ingestor = VectorStoreIngestor(persist_dir=persist_dir)
    print(f"✅ VectorStoreIngestor criado")
    print(f"   - Model: {ingestor.embedding_model_name}")
except Exception as e:
    print(f"❌ ERRO ao criar ingestor: {e}")
    import traceback
    traceback.print_exc()

📁 Persist dir: rag/vector_store/db/chroma
📁 Diretório existe? True
✅ VectorStoreIngestor criado
   - Model: sentence-transformers/all-MiniLM-L6-v2


## 4. Carregar Vector Store

In [24]:
try:
    vectorstore = ingestor.load_vectorstore()
    doc_count = vectorstore._collection.count()
    print(f"✅ Vector store carregado")
    print(f"   - Documentos: {doc_count}")
    
    if doc_count == 0:
        print("⚠️ Vector store vazio!")
except Exception as e:
    print(f"❌ ERRO ao carregar vector store: {e}")
    import traceback
    traceback.print_exc()

✅ Vector store carregado
   - Documentos: 385


## 5. Teste de Busca

In [28]:
if doc_count > 0:
    query = "Fale de introducao a web"
    print(f"🔍 Buscando: {query}")
    
    try:
        results = vectorstore.similarity_search(query, k=3)
        print(f"\n✅ Busca concluída!")
        print(f"   - Resultados encontrados: {len(results)}")
        
        for i, doc in enumerate(results, 1):
            print(f"\n📄 Resultado {i}:")
            print(f"   Página: {doc.metadata.get('page', 'N/A')}")
            print(f"   Conteúdo: {doc.page_content[:150]}...")
    except Exception as e:
        print(f"❌ ERRO na busca: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Não é possível testar busca - vector store vazio")

🔍 Buscando: Fale de introducao a web

✅ Busca concluída!
   - Resultados encontrados: 3

📄 Resultado 1:
   Página: 67
   Conteúdo: . 3 – EMENTA: A disciplina aborda a implementação de interfaces Web com experiência rica do usuário, estruturação correta e otimizações. O componente ...

📄 Resultado 2:
   Página: 68
   Conteúdo: Projeto Pedagógico do Curso Bacharelado em Engenharia de Software Modalidade: presencial 66 IFSP SÃO CARLOS BACHARELADO EM ENGENHARIA DE SOFTWARE BONA...

📄 Resultado 3:
   Página: 103
   Conteúdo: . 3 – EMENTA: A disciplina apresenta as tecnologias atuais para o desenvolvimento de aplicações Web que executam no servidor. O componente curricular ...


## 6. Teste de Ingestão (se necessário)

In [29]:
if doc_count == 0:
    print("🔄 Iniciando ingestão de dados...")
    try:
        vectorstore = ingestor.ingest_pdf(file_path)
        new_count = vectorstore._collection.count()
        print(f"✅ Ingestão concluída!")
        print(f"   - Documentos ingeridos: {new_count}")
    except Exception as e:
        print(f"❌ ERRO na ingestão: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭️ Pulando ingestão - dados já existem")

⏭️ Pulando ingestão - dados já existem


## 7. Verificar Estrutura de Dados

In [30]:
if doc_count > 0:
    print("📊 Analisando estrutura dos dados...")
    
    # Busca um documento qualquer
    sample_results = vectorstore.similarity_search("teste", k=1)
    
    if sample_results:
        sample = sample_results[0]
        print(f"\n✅ Estrutura do documento:")
        print(f"   Tipo: {type(sample)}")
        print(f"   Metadata keys: {list(sample.metadata.keys())}")
        print(f"   Tamanho do conteúdo: {len(sample.page_content)} caracteres")
        print(f"\n📝 Metadata completo:")
        for key, value in sample.metadata.items():
            print(f"   - {key}: {value}")

📊 Analisando estrutura dos dados...

✅ Estrutura do documento:
   Tipo: <class 'langchain_core.documents.base.Document'>
   Metadata keys: ['title', 'producer', 'char_count', 'modDate', 'moddate', 'source', 'chunk_method', 'total_pages', 'format', 'page', 'author', 'creationDate', 'subject', 'file_path', 'creationdate', 'keywords', 'trapped', 'creator']
   Tamanho do conteúdo: 948 caracteres

📝 Metadata completo:
   - title: 
   - producer: PyPDF2
   - char_count: 1247
   - modDate: 
   - moddate: 
   - source: rag/data/documents/PPCENGSOFTWARE.pdf
   - chunk_method: smart_pdf_processor
   - total_pages: 151
   - format: PDF 1.3
   - page: 110
   - author: 
   - creationDate: 
   - subject: 
   - file_path: rag/data/documents/PPCENGSOFTWARE.pdf
   - creationdate: 
   - keywords: 
   - trapped: 
   - creator: 


## 8. Diagnóstico Final

In [31]:
print("\n" + "="*50)
print("📋 DIAGNÓSTICO FINAL")
print("="*50)

checks = [
    ("Arquivo PDF existe", os.path.exists(file_path)),
    ("PdfProcessor funciona", 'processor' in locals()),
    ("Chunks gerados", 'chunks' in locals() and len(chunks) > 0),
    ("VectorStoreIngestor criado", 'ingestor' in locals()),
    ("Vector store carregado", 'vectorstore' in locals()),
    ("Documentos no banco", doc_count > 0),
    ("Busca funciona", 'results' in locals() and len(results) > 0)
]

for check, status in checks:
    icon = "✅" if status else "❌"
    print(f"{icon} {check}")

print("\n" + "="*50)


📋 DIAGNÓSTICO FINAL
✅ Arquivo PDF existe
✅ PdfProcessor funciona
✅ Chunks gerados
✅ VectorStoreIngestor criado
✅ Vector store carregado
✅ Documentos no banco
✅ Busca funciona

